In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import recall_score, accuracy_score, f1_score, precision_score, confusion_matrix

In [2]:
var_cat = [
    'Madrugada',
    'Dia_do_Mes',
    'Dia_de_Pagamento',
    'Valor_Redondo',
    'Status_Pendente',
    'Mesmo_Banco',
    'DiaDaSemana',
    'FimDeSemana',
    'Horario_Comercial',
    'TipoChave',
    'Status'
]

var_num = [
    'Valor', 
    'Hora'
]

var_ignored = [
    'EndToEndId',
    'DataHora_Tratada',
    'Anomalia'
]

In [3]:
df = df = pd.read_csv("../data/base_tratada.csv")
df['Hora'] = df['Hora'].clip(lower=12)
df['Valor'] = df['Valor'].clip(upper=4000)

X = df.copy()

In [4]:
colunas_numericas = var_num
colunas_categoricas = var_cat

preprocessor = ColumnTransformer(
    transformers=[
        # StandardScaler é vital para o DBSCAN e ajuda o Isolation Forest
        ('num', StandardScaler(), colunas_numericas),
        # OneHotEncoder transforma categorias em 0 e 1
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), colunas_categoricas)
    ]
)

In [5]:
# ==========================================
# PIPELINE 1: ISOLATION FOREST (A melhor opção)
# ==========================================
# O Isolation Forest tem o método .predict(), então podemos embutir 
# o modelo perfeitamente dentro do Pipeline do scikit-learn.

pipeline_if = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('modelo', IsolationForest(
        n_estimators=100, 
        contamination=0.1, # Sua proporção de 1% (100 em 10.000)
        random_state=42
    ))
])

# Treina e já prevê se é anomalia (-1) ou normal (1)
df['anomalia_IF'] = pipeline_if.fit_predict(X)

In [6]:
df['Anomalia'].value_counts()

Anomalia
0    9500
1     500
Name: count, dtype: int64

In [7]:
df['anomalia_IF'].value_counts()

anomalia_IF
 1    9000
-1    1000
Name: count, dtype: int64

In [8]:
df['anomalia_IF'] = np.where(df['anomalia_IF'] == -1, 1, 0)

In [9]:
y_real = df['Anomalia']
y_pred = df['anomalia_IF']

In [ ]:
recall = recall_score(y_real, y_pred, pos_label=1)
precisao = precision_score(y_real, y_pred, pos_label=1)
f1 = f1_score(y_real, y_pred, pos_label=1)
acuracia = accuracy_score(y_real, y_pred)

print(f"Recall: {recall:.2%}")
print(f"Precisão: {precisao:.2%}")
print(f"F1-Score: {f1:.2%}")
print(f"Acurácia: {acuracia:.2%}")

In [12]:
matriz = confusion_matrix(y_real, y_pred)
print("\nMatriz de Confusão:")
print(matriz)


Matriz de Confusão:
[[8537  963]
 [ 463   37]]


In [13]:
perfil_fraude = df.groupby('Anomalia')[colunas_numericas].describe()
print(perfil_fraude)

           Valor                                                      \
           count         mean          std   min        25%      50%   
Anomalia                                                               
0         9500.0  2453.622808  1312.287515  5.62  1301.3475  2619.27   
1          500.0  1439.018000  1010.241815  0.00   619.8300  1373.05   

                              Hora                                         \
               75%     max   count       mean       std   min   25%   50%   
Anomalia                                                                    
0         3817.755  4000.0  9500.0  14.564632  3.570612  12.0  12.0  12.0   
1         2113.960  4000.0   500.0  19.564000  3.068670  12.0  18.0  20.0   

                      
           75%   max  
Anomalia              
0         17.0  23.0  
1         22.0  23.0  


In [14]:
perfil_fraude = df.groupby('anomalia_IF')[colunas_numericas].describe()
print(perfil_fraude)

              Valor                                                      \
              count         mean          std  min        25%       50%   
anomalia_IF                                                               
0            9000.0  2413.047744  1308.570450  0.0  1264.4025  2530.740   
1            1000.0  2311.495980  1392.703951  0.0  1002.1050  2378.345   

                                  Hora                                         \
                   75%     max   count       mean       std   min   25%   50%   
anomalia_IF                                                                     
0            3760.7875  4000.0  9000.0  14.871667  3.702499  12.0  12.0  12.0   
1            3794.2150  4000.0  1000.0  14.301000  3.746256  12.0  12.0  12.0   

                         
              75%   max  
anomalia_IF              
0            18.0  23.0  
1            16.0  23.0  


In [15]:
# ==========================================
# PIPELINE 2: DBSCAN 
# ==========================================
# O DBSCAN é transdutivo (não tem método .predict() para novos dados isolados).
# Portanto, a melhor prática é usar o pipeline apenas para transformar os dados,
# e aplicar o modelo no dado transformado logo em seguida.

# Passo A: Transformar os dados
X_processado_dbscan = preprocessor.fit_transform(X)

# Passo B: Instanciar e treinar o DBSCAN
# ATENÇÃO: Como usamos OneHotEncoder (0 e 1), o 'eps' precisará de muito ajuste 
# manual, pois a distância Euclidiana com variáveis binárias fica distorcida.
modelo_dbscan = DBSCAN(
    eps=0.5,        # Você terá que testar exaustivamente esse valor
    min_samples=3   # Mínimo de pontos para formar um agrupamento "normal"
)

df['cluster_DBSCAN'] = modelo_dbscan.fit_predict(X_processado_dbscan)

In [16]:
df['anomalia_DBSCAN'] = np.where(df['cluster_DBSCAN'] == -1, 1, 0)

In [17]:
y_pred_dbscan = df['anomalia_DBSCAN']
recall = recall_score(y_real, y_pred_dbscan, pos_label=1)
precisao = precision_score(y_real, y_pred_dbscan, pos_label=1)
f1 = f1_score(y_real, y_pred_dbscan, pos_label=1)
acuracia = accuracy_score(y_real, y_pred_dbscan)

print(f"Recall: {recall:.2%}")
print(f"Precisão: {precisao:.2%}")
print(f"F1-Score: {f1:.2%}")
print(f"Acurácia: {acuracia:.2%}")

Recall: 96.00%
Precisão: 5.50%
F1-Score: 10.41%
Acurácia: 17.36%
